<table align="left"><tr><td>
<a href="https://colab.research.google.com/github/kikim6114/nlp2026/blob/main/05.Language_Model-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="코랩에서 실행하기"/></a>
</td></tr></table>

In [ ]:
# Colab이나 Kaggle을 사용하는 경우가 아니면, 이 cell을 모두 주석화하여 skip할 것.
import os

repository_name = 'nlp2026'
repository_url = f'https://github.com/kikim6114/{repository_name}.git'

# 항상 루트 경로(/content/)로 이동 후 확인
%cd /content/

if not os.path.exists(repository_name):
    !git clone {repository_url}
    print(f"{repository_name} 클론 완료")
else:
    print(f"{repository_name} 폴더가 이미 존재합니다. 클론을 건너뜁니다.")
%cd nlp2026

<font face="Times New Roman" size=7 color='blue'>5. 언어 모델(Language Model)-2<font>

[알림] 이 코드는 Bengio et al(2003)의 논문을 기반으로 작성되었습니다.<br>
[Bengio et al 2003, A Neural Probabilistic Language Model](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf)."

## 순전파 신경망 언어 모델(Feedforward Neural Language Model)

In [ ]:
data = open('names.txt').read().splitlines()
data[:10]

In [ ]:
token_to_index = {tok: i for i, tok in enumerate('abcdefghijklmnopqrstuvwxyz')}
token_to_index['<s>'] = 26
token_to_index['</s>'] = 27
index_to_token = {i: tok for tok, i in token_to_index.items()}

#### 데이터셋 구성

- 데이터셋은 $x, y$ 쌍으로 구성된다.
- 여기서 $x$는 $(n-1)$개의 토큰으로 이루어진 문맥(context)이고, $y$는 다음 토큰.

In [ ]:
context = [token_to_index['<s>']] * context_size
tokens = list('kikim') + ['</s>']
tokens

In [ ]:
import torch

context_size = 5

def build_dataset(data):
    X, Y = [], []
    for item in data:
        context = [token_to_index['<s>']] * context_size
        tokens = list(item) + ['</s>']
        for token in tokens:
            X.append(context)
            Y.append(token_to_index[token])
            context = context[1:] + [token_to_index[token]]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

# 학습(train), 검증(dev), 테스트(test)로 분할
import random
random.seed(123)
random.shuffle(data)

n1 = int(0.8 * len(data))
n2 = int(0.9 * len(data))

X_train, Y_train = build_dataset(data[:n1])
X_dev, Y_dev = build_dataset(data[n1:n2])
X_test, Y_test = build_dataset(data[n2:])

X_train.shape, Y_train.shape

### 모델 정의

In [ ]:
import torch.nn as nn

class MLPLM(nn.Module):
    def __init__(self, vocab_size, context_size, embedding_size, hidden_size):
        super(MLPLM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_size)
        self.fc1 = nn.Linear(context_size * embedding_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x = self.embedding(x)       # (batch_size, context_size, hidden_size)
        x = x.view(x.shape[0], -1)  # (batch_size, context_size * hidden_size)
        x = torch.relu(self.fc1(x)) # (batch_size, hidden_size)
        x = self.fc2(x)             # (batch_size, vocab_size)
        return x



In [ ]:
model = MLPLM(len(token_to_index), context_size, 64, 64)

x = X_train[:2]
x

In [ ]:
model.forward(x).shape

### 학습

In [ ]:
import torch.optim as optim

model = MLPLM(len(token_to_index), context_size, 64, 64)
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters())}")

# 하이퍼파라미터
learning_rate = 0.001
num_epochs = 10
batch_size = 32

# 손실 함수와 옵티마이저
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# 학습 루프
for epoch in range(num_epochs):
    # 데이터 순서를 무작위로 섞는다
    perm = torch.randperm(len(X_train))
    X_train = X_train[perm]
    Y_train = Y_train[perm]

    model.train()
    total_loss = 0
    for i in range(0, len(X_train), batch_size):
        X_batch = X_train[i:i+batch_size]
        Y_batch = Y_train[i:i+batch_size]

        # 순전파(forward pass)
        outputs = model(X_batch)
        loss = criterion(outputs, Y_batch)

        # 역전파(backward pass) 및 파라미터 업데이트
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / (len(X_train) // batch_size)
    print(f'에폭 [{epoch+1}/{num_epochs}], 손실: {avg_loss:.4f}')


### 생성

In [ ]:
# 모델에서 시퀀스를 샘플링
def sample(model, context, max_length=100):
    model.eval()
    output = []
    with torch.no_grad():
        context = torch.tensor(context).unsqueeze(0)
        for i in range(max_length):
            logits = model(context)
            probs = torch.softmax(logits, dim=-1)
            token = torch.multinomial(probs, num_samples=1)
            context = torch.cat([context[:, 1:], token], dim=1)

            output.append(index_to_token[token.item()])
            if index_to_token[token.item()] == '</s>':
                return ''.join(output)
    return ''.join(output)


In [ ]:
for i in range(10):
    print(sample(model, [token_to_index['<s>']] * context_size))

### 조건부 생성(Conditional Generation)

In [ ]:
prompt = 's'
for i in range(10):
    out = sample(model, ([token_to_index['<s>']] * (context_size-len(prompt))) + [token_to_index[c] for c in prompt])
    print(prompt + out)